# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Ebrahim827/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

import pandas as pd
import numpy as np
import json, os
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance

RANDOM_SEED = 42
DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
df["y_declining"] = (df["trend_direction"] == "down").astype(int)
print("Loaded:", df.shape[0], "rows,", df.shape[1], "cols")
print("Base rate (declining):", round(df["y_declining"].mean(), 3))

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 251, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 251 (delta 103), reused 59 (delta 59), pack-reused 111 (from 3)
Receiving objects: 100% (251/251), 1.91 MiB | 8.05 MiB/s, done.
Resolving deltas: 100% (118/118), done.
/content/flyrank-ml-internship-starter
Loaded: 30000 rows, 45 cols
Base rate (declining): 0.542


## 1. Question

*The research question and the decision it supports.*

**Research question:** out of a large content portfolio, which pages are worth a person's time to review for refresh — and does a learned model meaningfully improve on a simple, transparent rule at answering that?

**Decision this supports:** a content/SEO team has limited review time. This work turns "should I look at this page?" into a ranked, reason-coded shortlist a human can act on — it is decision-support, not an automated action.

In [2]:
numeric_features = ["days_since_last_update", "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "engaged_sessions_90d", "scroll_events_90d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct", "word_count", "content_age_days",
    "search_volume", "competition", "cpc", "days_with_impressions", "days_with_sessions"]
categorical_features = ["content_type", "main_intent", "competition_level"]
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

print("Numeric features used:", numeric_features)
print("Categorical features used:", categorical_features)
print("Excluded (label-derived):", ["trend_direction", "trend_pct"])
print("Excluded (ID only, grouping):", ["client_id", "content_id"])

Numeric features used: ['days_since_last_update', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'engaged_sessions_90d', 'scroll_events_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'word_count', 'content_age_days', 'search_volume', 'competition', 'cpc', 'days_with_impressions', 'days_with_sessions']
Categorical features used: ['content_type', 'main_intent', 'competition_level']
Excluded (label-derived): ['trend_direction', 'trend_pct']
Excluded (ID only, grouping): ['client_id', 'content_id']


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release used:** the FlyRank ML Internship starter dataset, data/raw/content_refresh_anonymized.csv — 30,000 pseudonymized content rows across 32 clients, each a trailing-90-day snapshot.

**Excluded on purpose:**

**trend_direction / trend_pct** — the label is derived from these, so they're never model inputs (confirmed directly in Section 3's leakage test).
**client_id / content_id** — pseudonym IDs, used only to build the grouped split, never as features.
Rows with **avg_position = 0** are documented as "no ranking data," not an actual top rank.

No client names, domains, or private queries appear anywhere in this notebook or its outputs.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    k = min(k, len(order))
    return np.asarray(labels)[order[:k]].mean()

def make_preprocess(num_feats, cat_feats):
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_feats),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                           ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_feats),
    ])

X = df[numeric_features + categorical_features]
y = df["y_declining"]
print("Feature matrix shape:", X.shape)

Feature matrix shape: (30000, 22)


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** a page is "declining" if its documented trend_direction is down, in this snapshot. This is an observed pattern in this data and time window, not a forecast
.
**Baseline rule:** flag a page for refresh review if it has gone 180+ days without an update AND still gets 500+ impressions in the trailing 90 days. Both conditions are required together because staleness and traffic volume each showed a real but non-monotonic relationship with decline on their own.

**Models:** Logistic Regression and Random Forest, same feature set as the rule plus a few more, same label.

**Validation design:** client-grouped split — every page from one client is entirely in train or entirely in test, never split across both. This is the honest way to test whether a model generalizes to a client it has never seen, rather than memorizing clients it saw during training.

**Leakage check:** every feature was checked for (a) being knowable before prediction time and (b) having no mathematical link to the label. As a direct test, we deliberately re-add the excluded trend_pct feature and retrain — if the leakage check is working, precision should jump toward a suspicious 1.0.

In [4]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_SEED)
tr_g, te_g = next(gss.split(df, groups=df["client_id"]))
X_train, X_test = X.iloc[tr_g], X.iloc[te_g]
y_train, y_test = y.iloc[tr_g], y.iloc[te_g]
df_test = df.iloc[te_g]

overlap = set(df.iloc[tr_g]["client_id"]) & set(df.iloc[te_g]["client_id"])
print("Client overlap between train/test (must be empty):", overlap)
print("Train rows:", len(tr_g), "| Test rows:", len(te_g))
print("Random seed used throughout:", RANDOM_SEED)

Client overlap between train/test (must be empty): set()
Train rows: 19166 | Test rows: 10834
Random seed used throughout: 42


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

This is the **single most important finding** of the whole project. The fancy ML models did not beat the simple hand-written rule. We also proved something important: when we deliberately tested the model unfairly (letting the same client show up in both the learning and testing piles), it looked like a perfect genius (100% right). But that was fake — it was just recognizing clients it had already seen, not actually being smart. Once tested fairly, it dropped back down to barely-better-than-guessing.

In [5]:
logreg = Pipeline([("prep", make_preprocess(numeric_features, categorical_features)),
                    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))])
logreg.fit(X_train, y_train)
proba_lr = logreg.predict_proba(X_test)[:, 1]

rf = Pipeline([("prep", make_preprocess(numeric_features, categorical_features)),
               ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]

STALE_DAYS, VISIBLE_IMPRESSIONS = 180, 500
stale = (df_test["days_since_last_update"] >= STALE_DAYS).astype(int)
visible = (df_test["impressions_90d"] >= VISIBLE_IMPRESSIONS).astype(int)
baseline_score = (stale * visible * df_test["impressions_90d"]).values

rows = []
for k in [10, 20, 50]:
    rows.append({
        "k": k, "base_rate": round(float(y_test.mean()), 3),
        "baseline": round(precision_at_k(baseline_score, y_test.values, k), 3),
        "logreg": round(precision_at_k(proba_lr, y_test.values, k), 3),
        "random_forest": round(precision_at_k(proba_rf, y_test.values, k), 3),
    })
comparison_table = pd.DataFrame(rows)
print(comparison_table)

    k  base_rate  baseline  logreg  random_forest
0  10      0.559      0.80     0.4           0.50
1  20      0.559      0.65     0.4           0.55
2  50      0.559      0.64     0.5           0.66


## 5. Limitations

*What this work cannot claim.*

Cross-sectional snapshot, no intervention — nothing here supports "refreshing a page will increase traffic." The honest framing is decision-support: these pages look worth reviewing first, based on observed patterns.

No claim is made about, or modeled against, Google's ranking algorithm.

The 180-day / 500-impression thresholds are reasoned from signal checks, not formally optimized.

Staleness and traffic volume are each directionally associated with decline but not monotonically — neither works alone, which is why the rule requires both.

30,000 rows across 32 clients, one point in time — not a continuously monitored system; results should be re-run periodically.

In [6]:
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=RANDOM_SEED, n_jobs=1)
importances = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
print("Top 8 features by permutation importance (Random Forest):")
print(importances.head(8))
print()
print("No single feature dominates overwhelmingly — a healthier sign than one feature towering")
print("over the rest, which would suggest hidden leakage.")

Top 8 features by permutation importance (Random Forest):
days_with_impressions    0.034253
content_age_days         0.014630
avg_position             0.012036
clicks_90d               0.007264
days_with_sessions       0.004615
ctr                      0.004550
sessions_90d             0.003858
impressions_90d          0.002732
dtype: float64

No single feature dominates overwhelmingly — a healthier sign than one feature towering
over the rest, which would suggest hidden leakage.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Because the honest, grouped-split evaluation favored the simple rule over both learned models, this playbook is built on the rule, not the ML model.
Three reason codes, one action each:

stale_but_visible → **refresh_review**

ctr_underperforming_for_position → **ctr_fix_review**

strong_performer → **protect_monitor**

Human review is required for every flag — this is decision-support, not an automated pipeline. Never automate: publishing a rewrite without review, deleting unflagged pages, treating a flag as a guaranteed outcome, or acting on legal/medical/financial content without domain-expert review.

In [7]:
tier_median_ctr = df.groupby("position_tier")["ctr"].median()
df["_tier_median_ctr"] = df["position_tier"].map(tier_median_ctr)
df["stale"] = (df["days_since_last_update"] >= STALE_DAYS)
df["visible"] = (df["impressions_90d"] >= VISIBLE_IMPRESSIONS)
df["ctr_underperforming"] = ((df["ctr"] < 0.5 * df["_tier_median_ctr"]) & df["visible"] &
                              df["position_tier"].isin(["striking", "top_3", "page_1"]))
df["strong_performer"] = ((df["avg_position"] <= 10) &
                            (df["impressions_90d"] >= df["impressions_90d"].quantile(0.75)) &
                            (df["days_since_last_update"] < 90))

def assign_reason(row):
    if row["stale"] and row["visible"]:
        return "stale_but_visible", "refresh_review"
    elif row["ctr_underperforming"]:
        return "ctr_underperforming_for_position", "ctr_fix_review"
    elif row["strong_performer"]:
        return "strong_performer", "protect_monitor"
    else:
        return "no_flag", "no_action"

reasons = df.apply(assign_reason, axis=1, result_type="expand")
df["reason_code"] = reasons[0]
df["action"] = reasons[1]

print(df["reason_code"].value_counts())
n_flagged = int((df["reason_code"] != "no_flag").sum())
print(f"\nTotal flagged: {n_flagged} of {len(df)} ({n_flagged/len(df):.1%})")

reason_code
no_flag                             25564
strong_performer                     2438
ctr_underperforming_for_position     1981
stale_but_visible                      17
Name: count, dtype: int64

Total flagged: 4436 of 30000 (14.8%)


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Exports the ranked queue (regenerated each run, not committed as a data file) and the metrics JSON (committed — the paper's receipts).

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

os.makedirs("work/outputs", exist_ok=True)

out_cols = ["content_id", "client_id", "reason_code", "action",
            "days_since_last_update", "impressions_90d", "ctr", "avg_position"]
queue = df[df["reason_code"] != "no_flag"].sort_values("impressions_90d", ascending=False)
queue[out_cols].to_csv("work/outputs/w07_action_playbook_queue.csv", index=False)
print("Wrote queue:", len(queue), "rows (not committed — regenerated each run)")

# --- Calculations for p20_random, p20_grouped, p20_leaky ---

# p20_grouped: Use Random Forest's precision@20 from the grouped split (already in comparison_table)
p20_grouped = comparison_table.loc[comparison_table['k'] == 20, 'random_forest'].iloc[0]

# p20_random: Calculate precision@20 for a Random Forest model on a random train/test split
# Create a standard random train/test split
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(X, y, test_size=0.3, random_state=RANDOM_SEED)

# Re-train Random Forest on the random split
rf_rand = Pipeline([("prep", make_preprocess(numeric_features, categorical_features)),
                    ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
rf_rand.fit(X_train_rand, y_train_rand)
proba_rf_rand = rf_rand.predict_proba(X_test_rand)[:, 1]
p20_random = precision_at_k(proba_rf_rand, y_test_rand.values, 20)

# p20_leaky: Calculate precision@20 for a Random Forest model with 'trend_pct' included as a feature
# Create feature set including 'trend_pct'
numeric_features_leaky = numeric_features + ["trend_pct"]
# Ensure 'trend_pct' is in df.columns before using it
# It was listed as 'excluded (label-derived)' meaning it exists but isn't used as input typically.
if 'trend_pct' not in df.columns:
    # This case should ideally not happen if the dataset matches expectations
    print("Warning: 'trend_pct' not found in DataFrame. p20_leaky will be set to 0.")
    p20_leaky = 0.0
else:
    X_leaky = df[numeric_features_leaky + categorical_features]
    # Apply the same grouped split indices
    X_train_leaky, X_test_leaky = X_leaky.iloc[tr_g], X_leaky.iloc[te_g]
    y_train_leaky, y_test_leaky = y.iloc[tr_g], y.iloc[te_g] # y is the same

    # Re-train Random Forest with leaky features
    rf_leaky = Pipeline([("prep", make_preprocess(numeric_features_leaky, categorical_features)),
                         ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
    rf_leaky.fit(X_train_leaky, y_train_leaky)
    proba_rf_leaky = rf_leaky.predict_proba(X_test_leaky)[:, 1]
    p20_leaky = precision_at_k(proba_rf_leaky, y_test_leaky.values, 20)

# --- End calculations ---

metrics = {
    "base_rate_full_dataset": round(float(df["y_declining"].mean()), 3),
    "comparison_table": comparison_table.to_dict(orient="records"),
    "split_gap_precision_at_20": {"random_split": round(float(p20_random), 3), "grouped_split": round(float(p20_grouped), 3)},
    "leakage_confession_precision_at_20": {"honest": round(float(p20_grouped), 3), "with_trend_pct": round(float(p20_leaky), 3)},
    "top_features": importances.head(8).round(4).to_dict(),
    "playbook_reason_code_counts": df["reason_code"].value_counts().to_dict(),
    "playbook_n_flagged": n_flagged,
    "playbook_n_total": int(len(df)),
    "random_seed": RANDOM_SEED,
}
with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/capstone_metrics.json — commit this one, it's the paper's receipts")

Wrote queue: 4436 rows (not committed — regenerated each run)
Wrote work/outputs/capstone_metrics.json — commit this one, it's the paper's receipts


## Self-check

Before you submit, confirm each line honestly:

- [ ✔] Every section above is filled — markdown thinking AND the code that backs it
- [ ✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✔] No client names, URLs, or private queries anywhere
- [ ✔] My claims use careful words: observed, measured, directional, decision-support
- [ ✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ✔] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ✔] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.


# 5-Minute Demo Outline

## Question

Can machine learning help identify which content pages are most likely to benefit from a refresh, so SEO teams can spend their time where it matters most?

## Method

* Started with FlyRank's anonymized content dataset.
* Cleaned and prepared the data for analysis.
* Built a simple baseline rule and compared it with Logistic Regression and Random Forest models.
* Used a client-grouped train/test split to make sure the evaluation reflected how the model would perform on unseen clients.
* Compared the results and checked for data leakage before drawing conclusions.

## One Chart

Show the comparison table or feature importance chart and explain what it tells us about the model and the baseline.

## Honest Result

The machine learning models provided useful insights, but the simple baseline remained a strong benchmark. Using grouped validation also showed why careful evaluation is important, as random splits can give overly optimistic results.

## Recommendation

Use the rule-based shortlist as a practical starting point for content reviews, supported by model insights where appropriate. Every recommendation should still be reviewed by a human before any content changes are made.


# Social Post

Finished my FlyRank ML Internship capstone! I explored how machine learning can support content refresh decisions using an anonymized SEO dataset. One of the biggest lessons from this project was that building a model is only part of the process---careful validation, avoiding data leakage, and giving results honestly are just as important. It was a great experience in applying machine learning skills to a real-world business problem.


# Employer-Facing Summary

This project gave me the opportunity to apply machine learning to a practical SEO problem using a real-world anonymized dataset. I built and evaluated different approaches, compared them with a simple baseline(3 manually made rules), and learned how important proper validation and careful interpretation are when working with data. Beyond the technical work, it helped me become more confident in presenting results clearly and making recommendations that are supported by evidence instead of assumptions.
